# Checkpoint 2

In checkpoint 1 we tried to predict calories from the food name and the serving size in grams. The calorie file only has one row per dish, so that model never looks at a photo. If the dish is new, there is nothing to look up.

For checkpoint 2 we switched to the part that actually needs a photo, which is figuring out what the dish is. Once we know the dish, we can just look up the calories in the table. We only used the 27 Food-101 dishes that already have a calorie row, not all 101.

The model we trained is logistic regression on features from a ResNet18 that was already trained on ImageNet. We did not update the ResNet weights. The training code is in `scripts/checkpoint2_baseline.py`. To run it again:

```bash
.venv/bin/python scripts/checkpoint2_baseline.py
```

The features are saved, so a second run only fits the classifier again.

## 1. Preprocessing pipeline

The raw images are Food-101 JPEGs. The long side is at most 512 pixels. The calorie table is `scripts/food_calories.json`.

This is what we actually did to the data:

- We only kept a class if its folder name is in the calorie table. All 27 names match a Food-101 class. We dropped the other 74 classes because we could not turn that prediction into calories yet.
- The calorie table has no duplicate names and no missing calorie values. `onion_rings` is missing grams. We do not use serving size as an input, so we left that blank and it does not change training. We still need a number there later if we want to report calories for onion rings.
- We open every image as RGB. There are no missing pixels to fill in.
- The ResNet18 ImageNet transform resizes the short side to 256, center crops to 224 by 224, scales pixels to between 0 and 1, then uses the ImageNet mean `[0.485, 0.456, 0.406]` and std `[0.229, 0.224, 0.225]`. Those numbers are fixed. We did not compute them from our train or test images.
- We keep ResNet18 frozen and run it in eval mode, so batch norm uses the ImageNet stats. We replace the last layer with an identity layer. The vector that comes out of global average pooling is what we train on.

We do not one hot encode the food name. The name is the label, not a feature. That is the change from checkpoint 1, where `food_item` was one of the inputs.

## 2. Model input

One sample is one photo of a dish.

- The tensor that goes into ResNet18 is `(3, 224, 224)`. That is 3 color channels and a 224 by 224 image.
- The data that goes into the classifier we actually trained looks like this: `X_train` is `(20250, 512)` and `y_train` is `(20250,)`. `X_test` is `(6750, 512)` and `y_test` is `(6750,)`. Each row of `X` is the 512 number feature vector from ResNet18. Each value in `y` is a class id from 0 to 26.

## 3. Train/test split

We use Food-101's official split, then drop any class that is not in the calorie table.

- Train: 750 images per class, 20,250 total.
- Test: 250 images per class, 6,750 total.
- No photo is in both sets. Every class has the same number of images, so the split is already stratified.
- The normalization numbers and the ResNet weights come from ImageNet, not from our photos.
- We only fit logistic regression on the train features. We picked `C=1.0` before looking at the test set. We do not have a validation split yet because we did not tune anything. For checkpoint 3 we should hold out some of the official train images for validation and leave this test set alone.

`model/data/preprocessing.py` would make a new 80/20 split from the class folders. That would put official test photos into training. We did not do that here.

## 4. Baseline model

This is a 27 class problem. Each photo gets one dish, and a photo cannot be two classes at once.

We use logistic regression, which puts a softmax over the 27 classes. It is a simple linear model on top of features from a network that was already trained. We wanted to see if those features already separate the dishes before we fine tune the network in `model/train.py` for checkpoint 3.

## 5. Training code and hyperparameters

The fit is in `fit_baseline` in `scripts/checkpoint2_baseline.py`:

```python
clf = LogisticRegression(
    C=1.0,
    l1_ratio=0.0,
    solver="lbfgs",
    max_iter=1000,
    random_state=42,
)
clf.fit(x_train, y_train)
```

- `C=1.0` controls how strong the penalty is. A smaller `C` means a stronger penalty.
- `l1_ratio=0.0` means the penalty is only L2, not L1.
- `solver="lbfgs"` is the optimizer. It finished in 800 iterations, which is under the `max_iter=1000` limit.
- `random_state=42` makes the fit come out the same if we run it again.

We do not update the ResNet18 weights.

## 6. Metrics

- Accuracy. Every class has 250 test images, so accuracy is just how often the top guess is the right dish.
- Macro F1. This averages F1 across classes with equal weight, so a dish we always get wrong does not get hidden by the easy ones. Precision and recall for each dish are in `results/checkpoint2/classification_report.txt`.
- Confusion matrix. This shows which dishes get mixed up. Accuracy alone does not show that.

If we always guessed the most common training class, accuracy would only be 3.7% (1 out of 27). We did not use RMSE or ROC-AUC as the main scores. We are predicting a class, not a calorie number, and we are not picking a cutoff on one score.

## 7. Test results

These numbers are from the official test split, not the training images.

| | |
|---|---|
| Majority class accuracy | 0.037 |
| Train accuracy | 0.874 |
| Test accuracy | 0.712 |
| Test macro F1 | 0.711 |

![Confusion matrix on the test set](../results/checkpoint2/confusion_matrix.png)

![Recall for each class](../results/checkpoint2/per_class_recall.png)

![Mistakes the model was sure about](../results/checkpoint2/high_confidence_mistakes.png)

## 8. Interpretation

Test accuracy is 71.2%. That is a lot better than 3.7%, which is what you would get from always guessing the same class. These are also photos the model did not train on. Macro F1 is basically the same as accuracy because every class has the same number of test images, and we did not totally fail any class.

Train accuracy is 87.4%, about 16 points higher than test. So the linear layer is overfitting a little. It is not underfitting. Both scores are way above chance, and the fit on the training data is pretty good. The gap means we should tune `C` on a validation split and then fine tune the rest of the network. It does not mean the features are useless.

The easy dishes look pretty different from everything else. Pizza has recall 0.86, onion rings 0.85, deviled eggs 0.84, and french fries 0.83. The hard ones look like each other:

- Ravioli is the worst class, with recall 0.50. 23 of the 250 lasagna photos get called ravioli, and 17 ravioli photos get called lasagna.
- Pulled pork sandwich and hamburger get mixed up both ways (31 and 29 test photos).
- Pork chop and baby back ribs get mixed up too (24 and 17).
- Samosa gets called spring rolls 17 times. Spaghetti bolognese gets called pad thai 19 times.

The mistakes the model was sure about are the same kind of mixup, plus some photos that have more than one food on the plate. One pulled pork photo is mostly onion rings, and the model says onion rings. The label is only one dish, but the picture has two. A frozen network is going to keep mixing those up until we train it on the differences, like the bun versus the filling, the shape of the pasta, or a rib versus a chop.

## 9. Next steps before the final project

1. Fine tune the network with `model/train.py` instead of keeping it frozen. That code already trains the new last layer first, then unfreezes the rest of the network at a lower learning rate. The dishes that look alike, like pasta, sandwiches, and fried meat, are the ones this linear model cannot tell apart.
2. Tune hyperparameters on a validation split taken only from the official train set. The first thing to try is `C` for this linear model. After we fine tune, we can also try different learning rates and different numbers of epochs where the backbone stays frozen. We should not touch the official test set until we want the final number.

After we predict the class, we can look up calories in `food_calories.json`. We need to fill in the missing onion rings serving size before we do that. Calorie numbers will also be off when a photo has more than one dish.

In [ ]:
import json
from pathlib import Path

summary = json.loads(Path("../results/checkpoint2/metrics.json").read_text())
print("X_train", tuple(summary["x_train_shape"]), "y_train", tuple(summary["y_train_shape"]))
print("X_test ", tuple(summary["x_test_shape"]), "y_test ", tuple(summary["y_test_shape"]))
print(f"train acc {summary['train_accuracy']:.3f}")
print(f"test acc  {summary['test_accuracy']:.3f}")
print(f"test macro F1 {summary['test_macro_f1']:.3f}")
print("top confusions:")
for row in summary["top_confusions"]:
    print(f"  {row['count']:3d}  {row['true']} -> {row['predicted']}")